In [0]:
from pyspark.sql.functions import col, when, count, round

# 1. Carrega a tabela silver
df_silver = spark.table('foodDelivery.silver.orders')

# 2. Cria agregação: total de pedidos vs total reembolsados
df_gold = df_silver.agg(
    count("*").alias("total_pedidos"),
    count(when(col("order_status") == "Reembolsado", True)).alias("total_reembolsos")
)

# 3. Calcular com precisão de duas casas decimais
df_gold_kpi = df_gold.withColumn(
    "taxa_reembolso", 
    round(col("total_reembolsos") / col("total_pedidos") * 100, 2)
)

# 4. Salvar na camada Gold
(df_gold_kpi.write
 .format('delta')
 .mode('overwrite')
 .saveAsTable('foodDelivery.gold.kpi_reembolso'))

dbutils.notebook.exit("SUCCESS")